In [ ]:
# Step 1: Install and Import
!pip install -q transformers datasets accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset

# Step 2: Load GPT-2
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 needs a padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# Step 3: Prepare Data (New Style: Cyberpunk Noir)
data = [
    {"text": "Rain lashed against the jagged skyline, reflecting neon blues onto the wet pavement."},
    {"text": "In the sprawl of Sector 7, shadows moved faster than the law could follow."},
    {"text": "His cybernetic eye flickered, scanning the crowd for a face that didn't belong."},
    {"text": "The air tasted of ozone and cheap synthetic tobacco in the underground club."},
    {"text": "Chrome limbs glinted under flickering streetlamps as the mercenaries waited."},
    {"text": "Memory chips were the only currency that mattered in the dark corners of the net."},
    {"text": "A glitched hologram flickered in the alleyway, offering dreams no one could afford."},
    {"text": "The sirens wailed in the distance, a constant heartbeat of the dying city."},
    {"text": "He plugged the data-spike into his neural port, feeling the cold rush of stolen secrets."},
    {"text": "The megastructures blocked out the sun, leaving the world in a perpetual twilight."}
]
dataset = Dataset.from_list(data)

def tokenize_func(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

tokenized_data = dataset.map(tokenize_func, batched=True)
tokenized_data = tokenized_data.map(lambda x: {"labels": x["input_ids"]}, batched=True)

# Step 4: Training Setup
training_args = TrainingArguments(
    output_dir="./gpt2-cyberpunk",
    num_train_epochs=10, # Increased epochs for better style adoption
    per_device_train_batch_size=2,
    logging_steps=1,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    remove_unused_columns=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    processing_class=tokenizer
)

# Step 5: Train and Generate
print("Starting fine-tuning with Cyberpunk Noir dataset...")
trainer.train()

print("\n--- Generating Text ---")
prompt = "The rain reflected"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output_tokens = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    no_repeat_ngram_size=2
)

print(tokenizer.decode(output_tokens[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting fine-tuning with Cyberpunk Noir dataset...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,8.596246
2,6.055322
3,3.992215
4,1.926253
5,1.205196
6,1.089767
7,0.627427
8,0.599557
9,0.650834
10,0.570603



--- Generating Text ---
The rain reflected the wind as the city moved into the alley, leaving the residents to their fate.
